In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🚀 Ingesting New 1872-2026 International Match Database...")

# 1. Load the core datasets
results = pd.read_csv('results.csv')
shootouts = pd.read_csv('shootouts.csv')
goalscorers = pd.read_csv('goalscorers.csv')

# Convert dates to datetime objects
results['date'] = pd.to_datetime(results['date'])
shootouts['date'] = pd.to_datetime(shootouts['date'])
goalscorers['date'] = pd.to_datetime(goalscorers['date'])

# 2. Filter for the Modern Era (Post-2018 to 2026 momentum tracking)
# While the data goes back to 1872, tactical relevance drops drastically. 
# We'll calculate a time-decay weight, but strip out ancient history for form metrics.
modern_matches = results[results['date'] >= '2018-01-01'].copy()

# 3. Calculate Time-Decay Factor
# Matches played closer to the current date (mid-2026) get exponentially higher weights.
max_date = modern_matches['date'].max()
modern_matches['days_ago'] = (max_date - modern_matches['date']).dt.days
modern_matches['weight'] = np.exp(-0.0005 * modern_matches['days_ago']) # Smooth decay over 4-5 years

# 4. Feature Extraction: Track Historical Head-to-Head and Goal Proportions
def get_team_form_metrics(team_name, matches_df):
    """Calculates time-decayed scoring and defensive trends for a specific team."""
    home_games = matches_df[matches_df['home_team'] == team_name]
    away_games = matches_df[matches_df['away_team'] == team_name]
    
    if home_games.empty and away_games.empty:
        return {'weighted_goals_scored': 1.2, 'weighted_goals_conceded': 1.2}
    
    # Goals scored and conceded with weights applied
    home_scored = (home_games['home_score'] * home_games['weight']).sum()
    home_conceded = (home_games['away_score'] * home_games['weight']).sum()
    home_weight_total = home_games['weight'].sum()
    
    away_scored = (away_games['away_score'] * away_games['weight']).sum()
    away_conceded = (away_games['home_score'] * away_games['weight']).sum()
    away_weight_total = away_games['weight'].sum()
    
    total_weight = home_weight_total + away_weight_total
    
    weighted_avg_scored = ((home_scored + away_scored) / total_weight) if total_weight > 0 else 1.2
    weighted_avg_conceded = ((home_conceded + away_conceded) / total_weight) if total_weight > 0 else 1.2
    
    return {
        'weighted_goals_scored': round(weighted_avg_scored, 3),
        'weighted_goals_conceded': round(weighted_avg_conceded, 3)
    }

# Example: Check how the engine processes a specific team's modern tactical strength
sample_team = "Spain"
metrics = get_team_form_metrics(sample_team, modern_matches)
print(f"\n📊 Verified Ingestion for {sample_team}:")
print(f" -> Time-Decayed Expected Scored per Match: {metrics['weighted_goals_scored']}")
print(f" -> Time-Decayed Expected Conceded per Match: {metrics['weighted_goals_conceded']}")

🚀 Ingesting New 1872-2026 International Match Database...

📊 Verified Ingestion for Spain:
 -> Time-Decayed Expected Scored per Match: 2.293
 -> Time-Decayed Expected Conceded per Match: 0.76
